In [2]:
from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from PIL import Image
from optuna.terminator.improvement.emmr import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from torch import nn
from torch.utils.data import Dataset
from torchvision import models
from tqdm.auto import tqdm
import cv2

In [3]:
class BeerDataset(Dataset):
    PATH_PRIORITY = ("path", "aug_path", "cropped_path", "orig_path")

    def __init__(
        self,
        meta_csv: str | Path = "data/meta/full_dataset.csv",
        root: str | Path | None = None,
        transform=None,
        target_transform=None,
        return_info: bool = False,
        path_columns: tuple[str, ...] | None = None,
    ):
        self.meta_csv = Path(meta_csv)
        if not self.meta_csv.exists():
            raise FileNotFoundError(f"Metadata CSV not found: {self.meta_csv}")

        self.root = Path(root) if root is not None else Path(".")
        self.transform = transform
        self.target_transform = target_transform
        self.return_info = return_info

        df = pd.read_csv(self.meta_csv)
        required_cols = {"klass"}
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns in metadata: {sorted(missing)}")

        candidates = path_columns or self.PATH_PRIORITY
        available = tuple(col for col in candidates if col in df.columns)
        if not available:
            raise ValueError(
                "None of the provided path columns were found in the metadata."
            )

        resolved_path = df[available[0]].copy()
        for col in available[1:]:
            resolved_path = resolved_path.fillna(df[col])

        df = df.assign(resolved_path=resolved_path)
        df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)
        if df.empty:
            raise ValueError("No usable entries found in the metadata file.")

        df["resolved_path"] = df["resolved_path"].astype(str)
        self._path_column = "resolved_path"
        self._records = df

        classes = sorted(df["klass"].unique())
        self.class_to_idx = {klass: idx for idx, klass in enumerate(classes)}
        self.idx_to_class = {idx: klass for klass, idx in self.class_to_idx.items()}

    def __len__(self) -> int:
        return len(self._records)

    def _resolve_path(self, rel_path: str) -> Path:
        p = Path(rel_path)
        if p.is_absolute():
            return p
        return (self.root / p).resolve()

    def __getitem__(self, index: int):
        row = self._records.iloc[index]
        img_path = self._resolve_path(row[self._path_column])
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found on disk: {img_path}")

        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["klass"]]

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            label = self.target_transform(label)

        if self.return_info:
            info = row.to_dict() | {"img_path": str(img_path)}
            return image, label, info
        return image, label


In [4]:
import selectivesearch
class BeerRCNNDataset(BeerDataset):

    def __init__(
        self,
        *args,
        max_proposals: int = 2000,
        selective_search_kwargs: dict | None = None,
        min_size: int = 20,
        aspect_ratio_range: tuple[float, float] = (0.3, 3.0),
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.max_proposals = max_proposals
        self.ss_kwargs = selective_search_kwargs or dict(scale=500, sigma=0.8, min_size=50)
        self.min_size = min_size
        self.aspect_ratio_range = aspect_ratio_range

    def _generate_proposals(self, image_bgr: np.ndarray) -> np.ndarray:
        img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        _, regions = selectivesearch.selective_search(img_rgb, **self.ss_kwargs)

        boxes = []
        for r in regions:
            x, y, w, h = r["rect"]

            # handle 0 boxes
            if w <= 0 or h <= 0:
                continue

            # size
            if w < self.min_size or h < self.min_size:
                continue

            # ratio
            ratio = h / float(w)
            if not (self.aspect_ratio_range[0] <= ratio <= self.aspect_ratio_range[1]):
                continue

            boxes.append([x, y, x + w, y + h])

            if len(boxes) >= self.max_proposals:
                break

        return np.array(boxes, dtype=np.float32)

    def __getitem__(self, index: int):
        image_pil, label, info = super().__getitem__(index)
        image_bgr = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
        proposals = self._generate_proposals(image_bgr)
        return image_bgr, label, proposals, info

In [18]:

# ds = BeerRCNNDataset(
#     return_info=True,
#     max_proposals=2000,
# )
#
# img_bgr, label, props, info = ds[0]
# print("Image shape:", img_bgr.shape)
# print("Label:", label)
# print("Num proposals:", len(props))
#
# for (x1, y1, x2, y2) in props[:30].astype(int):
#     cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 1)

# cv2.imshow("Selective Search Proposals", img_bgr)
# cv2.waitKey(0)


In [19]:
# import numpy as np
# import cv2
# from tqdm import tqdm
# import torch
# import torch.nn as nn
# import torchvision.models as models
#
#
# class AlexNetFeatureExtractor(nn.Module):
#     def __init__(self, device="cpu"):
#         super().__init__()
#         alex = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
#         self.features = alex.features
#         self.avgpool = alex.avgpool
#         self.fc = nn.Sequential(*list(alex.classifier[:-1]))  # до fc7
#         self.device = device
#         self.eval()
#         self.to(device)
#         for p in self.parameters():
#             p.requires_grad = False
#
#     @torch.no_grad()
#     def forward(self, x):
#         x = self.features(x)
#         x = self.avgpool(x)
#         x = torch.flatten(x, 1)
#         x = self.fc(x)
#         return x
#
# def preprocess_crop(crop: np.ndarray) -> torch.Tensor:
#     crop = cv2.resize(crop, (227, 227))
#     crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
#     crop = crop.astype(np.float32) / 255.0
#     # Нормалізація під ImageNet
#     mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
#     std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
#     crop = (crop - mean) / std
#     crop_t = torch.from_numpy(crop.transpose(2, 0, 1)).unsqueeze(0)
#     return crop_t
#
# def extract_features(
#     meta_csv="data/meta/full_dataset.csv",
#     root="data/images",
#     output_dir="runs/rcnn/features",
#     max_props=300,
#     limit_per_image=100,
#     device="mps", #Change, according to your device
# ):
#     device = device or ("msp" if torch.cuda.is_available() else "cpu")
#     out = Path(output_dir)
#     out.mkdir(parents=True, exist_ok=True)
#
#     dataset = BeerRCNNDataset(
#         meta_csv=meta_csv,
#         root=root,
#         return_info=True,
#         max_proposals=max_props,
#     )
#
#     extractor = AlexNetFeatureExtractor(device=device)
#
#     X_feats, y_labels = [], []
#
#     print(f"[INFO] All images: {len(dataset)}")
#     for i in tqdm(range(len(dataset)), desc="Extracting CNN features"):
#         img_bgr, label, proposals, info = dataset[i]
#         H, W, _ = img_bgr.shape
#
#         for (x1, y1, x2, y2) in proposals[:limit_per_image].astype(int):
#             x1, y1 = max(0, x1), max(0, y1)
#             x2, y2 = min(W, x2), min(H, y2)
#             if x2 - x1 < 16 or y2 - y1 < 16:
#                 continue
#
#             pad = 16
#             x1p, y1p = max(0, x1 - pad), max(0, y1 - pad)
#             x2p, y2p = min(W, x2 + pad), min(H, y2 + pad)
#             crop = img_bgr[y1p:y2p, x1p:x2p]
#
#             if crop.size == 0:
#                 continue
#
#             crop_t = preprocess_crop(crop).to(device)
#
#             with torch.no_grad():
#                 feat = extractor(crop_t).cpu().numpy().squeeze()
#
#             X_feats.append(feat)
#             y_labels.append(label)
#
#     X_feats = np.stack(X_feats)
#     y_labels = np.array(y_labels)
#
#     np.save(out / "X_feats.npy", X_feats)
#     np.save(out / "y_labels.npy", y_labels)
#     print(f"✅ Saved: {X_feats.shape} features, {len(y_labels)} labels → {out}")
#
#
# extract_features()


In [20]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

mps


In [21]:
ds = BeerRCNNDataset(
    meta_csv="data/meta/full_dataset.csv",
    root="data/images",
    return_info=True,
    max_proposals=200,
)


In [22]:
class AlexNetFeatureExtractor(nn.Module):
    def __init__(self, device="cpu"):
        super().__init__()
        alex = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        self.features = alex.features
        self.avgpool = alex.avgpool
        self.fc = nn.Sequential(*list(alex.classifier[:-1]))  # up to fc7
        self.device = device
        self.eval()
        self.to(device)
        for p in self.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


In [23]:
def preprocess_crop(crop: np.ndarray) -> torch.Tensor:
    """
    torch.Tensor [1, 3, 227, 227]
    """
    crop = cv2.resize(crop, (227, 227))
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    crop = crop.astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    crop = (crop - mean) / std

    crop_t = torch.from_numpy(crop.transpose(2, 0, 1)).unsqueeze(0)
    return crop_t



In [24]:
def extract_features(
    meta_csv="data/meta/full_dataset.csv",
    root="data/images",
    output_dir="runs/rcnn/features",
    max_props=300,
    limit_per_image=100,
    device=device,
):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    dataset = BeerRCNNDataset(
        meta_csv=meta_csv,
        root=root,
        return_info=True,
        max_proposals=max_props,
    )

    extractor = AlexNetFeatureExtractor(device=device)

    X_feats, y_labels = [], []

    print(f"[INFO] Всього зображень: {len(dataset)}")
    for i in tqdm(range(len(dataset)), desc="Extracting CNN features"):
        img_bgr, label, proposals, info = dataset[i]
        H, W, _ = img_bgr.shape

        for (x1, y1, x2, y2) in proposals[:limit_per_image].astype(int):
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            if x2 - x1 < 16 or y2 - y1 < 16:
                continue

            pad = 16
            x1p, y1p = max(0, x1 - pad), max(0, y1 - pad)
            x2p, y2p = min(W, x2 + pad), min(H, y2 + pad)
            crop = img_bgr[y1p:y2p, x1p:x2p]

            if crop.size == 0:
                continue

            crop_t = preprocess_crop(crop).to(device)

            with torch.no_grad():
                feat = extractor(crop_t).cpu().numpy().squeeze()

            X_feats.append(feat)
            y_labels.append(label)

    X_feats = np.stack(X_feats)
    y_labels = np.array(y_labels)

    np.save(out / "X_feats.npy", X_feats)
    np.save(out / "y_labels.npy", y_labels)
    print(f" Saved: {X_feats.shape} features, {len(y_labels)} labels → {out}")


In [25]:
extract_features(
    meta_csv="data/meta/full_dataset.csv",
    root="",
    output_dir="runs/rcnn/features",
    max_props=300,
    limit_per_image=100,
    device=device,
)


[INFO] Всього зображень: 7405


Extracting CNN features:   0%|          | 0/7405 [00:00<?, ?it/s]

/Users/ivantyshchenko/Projects/Python/Neural-Beer/.venv/lib/python3.12/site-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


 Saved: (609315, 4096) features, 609315 labels → runs/rcnn/features


In [5]:
def load_features_featureset(
    features_dir="runs/rcnn/features",
    feats_name="X_feats.npy",
    labels_name="y_labels.npy",
    boxes_name="boxes.npy",
    image_ids_name="image_ids.npy",
):
    d = Path(features_dir)
    X = np.load(d / feats_name)
    y = np.load(d / labels_name)

    boxes, image_ids = None, None
    bx = d / boxes_name
    im = d / image_ids_name
    if bx.exists() and im.exists():
        boxes = np.load(bx)
        image_ids = np.load(im)

    print(f"[INFO] Loaded X: {X.shape}, y: {y.shape}, boxes: {None if boxes is None else boxes.shape}")
    return X, y, boxes, image_ids



X, y, boxes, image_ids = load_features_featureset(
    features_dir="runs/rcnn/features",
    feats_name="X_feats.npy",
    labels_name="y_labels.npy",
    boxes_name="boxes.npy",          # якщо не зберігав — залиш без цих файлів
    image_ids_name="image_ids.npy",  # якщо не зберігав — залиш без цих файлів
)


[INFO] Loaded X: (609315, 4096), y: (609315,), boxes: None


In [6]:
def build_svm_pipeline(C=1.0, max_iter=5000, use_prob=True):
    """
    OVR LinearSVC з optional калібровкою (sigmoid), загорнутий у Pipeline зі StandardScaler.
    """
    base = OneVsRestClassifier(
        LinearSVC(C=C, class_weight="balanced", dual=False, max_iter=max_iter)
    )
    if use_prob:
        clf = CalibratedClassifierCV(estimator=base, method="sigmoid", cv=3)
        return Pipeline([("scaler", StandardScaler(with_mean=True)), ("clf", clf)])
    else:
        return Pipeline([("scaler", StandardScaler(with_mean=True)), ("clf", base)])

print("✅ Pipeline builder ready")


✅ Pipeline builder ready


In [ ]:
def train_eval_svm(
    X, y, test_size=0.2, C=1.0, max_iter=5000, use_prob=True, model_dir="runs/rcnn/models"
):
    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_enc, test_size=test_size, stratify=y_enc, random_state=42
    )

    pipe = build_svm_pipeline(C=C, max_iter=max_iter, use_prob=use_prob)
    pipe.fit(X_tr, y_tr)

    y_pred = pipe.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1m = f1_score(y_te, y_pred, average="macro")

    print(f"[RESULT] Acc={acc:.4f}, F1-macro={f1m:.4f}\n")
    print("Classification report:")
    print(classification_report(y_te, y_pred, target_names=le.classes_.astype(str)))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_te, y_pred))

    md = Path(model_dir); md.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, md / "svm_ovr.pkl")
    joblib.dump(le,   md / "label_encoder.pkl")
    print(f"\n[SAVE] Model → {md/'svm_ovr.pkl'}")
    print(f"[SAVE] LabelEncoder → {md/'label_encoder.pkl'}")

    return pipe, le, (X_tr, X_te, y_tr, y_te)


# тренування базової OVR-моделі
svm_pipe, le, splits = train_eval_svm(
    X, y,
    test_size=0.2,
    C=1.0,
    max_iter=5000,
    use_prob=True,
    model_dir="runs/rcnn/models"
)


In [ ]:
def train_svm_per_class_with_hnm(
    X, y, classes: np.ndarray, C=1.0, max_iter=5000, K=2000, rounds=1
):
    """
    Повертає dict: class_id -> (scaler, svm)
    y має бути індексами класів (LabelEncoder.transform).
    K — скільки hard negatives добирати кожного раунду.
    """
    models = {}
    for c in classes:
        pos_idx = np.where(y == c)[0]
        neg_idx = np.where(y != c)[0]
        X_pos, X_neg = X[pos_idx], X[neg_idx]

        scaler = StandardScaler(with_mean=True)
        X_pos = scaler.fit_transform(X_pos)
        X_neg = scaler.transform(X_neg)

        svm = LinearSVC(C=C, class_weight="balanced", dual=False, max_iter=max_iter)
        # стартова фаза: всі neg
        X_train = np.vstack([X_pos, X_neg])
        y_train = np.hstack([np.ones(len(X_pos)), np.zeros(len(X_neg))])
        svm.fit(X_train, y_train)

        # HNM-раунди
        for r in range(rounds):
            scores = svm.decision_function(X_neg)
            # найважчі негативи — найбільші скорі
            hard_idx = np.argsort(scores)[::-1][:min(K, len(scores))]
            X_hard = X_neg[hard_idx]

            X_train = np.vstack([X_pos, X_hard])
            y_train = np.hstack([np.ones(len(X_pos)), np.zeros(len(X_hard))])
            svm.fit(X_train, y_train)

        models[int(c)] = (scaler, svm)
        print(f"[HNM] class={c}: pos={len(X_pos)}, neg_pool={len(X_neg)}, rounds={rounds}")
    return models


def save_hnm_models(models, label_encoder, model_dir="runs/rcnn/models_hnm"):
    md = Path(model_dir); md.mkdir(parents=True, exist_ok=True)
    payload = {"models": models, "label_encoder": label_encoder}
    joblib.dump(payload, md / "svm_hnm.pkl")
    print(f"[SAVE] HNM models → {md/'svm_hnm.pkl'}")


# запуск HNM-навчання (за бажанням)
y_enc = le.transform(y)
cls_ids = np.unique(y_enc)

hnm_models = train_svm_per_class_with_hnm(
    X, y_enc, classes=cls_ids,
    C=1.0, max_iter=5000, K=2000, rounds=1
)
save_hnm_models(hnm_models, le, model_dir="runs/rcnn/models_hnm")


In [ ]:
def predict_scores_ovr(pipe: Pipeline, le: LabelEncoder, Xq: np.ndarray):
    """
    Повертає:
      - y_pred_labels (argmax по класах, у вихідних індексах LabelEncoder.inverse_transform)
      - scores: predict_proba (якщо калібрований) або decision_function
    """
    clf = pipe.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        scores = pipe.predict_proba(Xq)  # (N, C)
    else:
        scores = pipe.decision_function(Xq)
        if scores.ndim == 1:  # binary випадок
            scores = scores[:, None]
    y_pred = pipe.predict(Xq)
    y_pred_labels = le.inverse_transform(y_pred)
    return y_pred_labels, scores

print("✅ Predict helpers ready")


In [ ]:
def iou_xyxy(a: np.ndarray, b: np.ndarray) -> float:
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-6
    return inter / union


def nms(boxes: np.ndarray, scores: np.ndarray, iou_thresh=0.3):
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        rest = order[1:]
        survivors = []
        for j in rest:
            if iou_xyxy(boxes[i], boxes[j]) <= iou_thresh:
                survivors.append(j)
        order = np.array(survivors, dtype=int) if survivors else np.array([], dtype=int)
    return keep


In [ ]:
def detections_with_nms_for_class(
    class_idx: int,
    scores: np.ndarray,     # (N, C)
    boxes: np.ndarray,      # (N, 4)
    image_ids: np.ndarray,  # (N,)
    thresh=0.5,
    iou_thresh=0.3
):
    mask = scores[:, class_idx] >= thresh
    if mask.sum() == 0:
        return []

    boxes_c = boxes[mask]
    scores_c = scores[mask, class_idx]
    ids_c = image_ids[mask]
    keep = nms(boxes_c, scores_c, iou_thresh=iou_thresh)

    out = []
    for k in keep:
        out.append((int(ids_c[k]), float(scores_c[k]), boxes_c[k].tolist()))
    return out


def run_detection_nms(
    pipe: Pipeline, le: LabelEncoder,
    X_all: np.ndarray, boxes: np.ndarray, image_ids: np.ndarray,
    score_thresh=0.6, iou_thresh=0.3
):
    _, scores = predict_scores_ovr(pipe, le, X_all)
    if scores.ndim == 1:  # edge-case для binary
        scores = np.stack([1.0 - scores, scores], axis=1)

    results = {}
    for cls_idx, cls_name in enumerate(le.classes_):
        dets = detections_with_nms_for_class(
            class_idx=cls_idx,
            scores=scores,
            boxes=boxes,
            image_ids=image_ids,
            thresh=score_thresh,
            iou_thresh=iou_thresh
        )
        results[str(cls_name)] = dets
        print(f"[NMS] class={cls_name}: kept {len(dets)} detections")
    return results


# якщо під час фіч-екстракції ти зберігав boxes/image_ids — можна отримати NMS-детекції:
if boxes is not None and image_ids is not None:
    dets = run_detection_nms(
        pipe=svm_pipe, le=le,
        X_all=X, boxes=boxes, image_ids=image_ids,
        score_thresh=0.6, iou_thresh=0.3
    )


In [ ]:
test_id = 88
per_image = {k: [d for d in v if d[0] == test_id] for k, v in dets.items()}
for cls, items in per_image.items():
    print(cls, "→", items[:5])
